In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.dummy import DummyClassifier

URL = ("https://raw.githubusercontent.com/justmarkham/"
       "pycon-2016-tutorial/master/data/sms.tsv")
sms = pd.read_csv(URL, sep="\t", header=None, names=["label", "message"])

X = sms["message"]
y = (sms["label"] == "spam").astype(int)
print(sms.shape, y.value_counts().to_dict(), round(y.mean(), 3))
# (5572, 2) {0: 4825, 1: 747} 0.134
print(sms.loc[sms["label"] == "spam", "message"].iloc[0][:80]) # lấy tin đầu + cắt chuỗi 80
# Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 8

(5572, 2) {0: 4825, 1: 747} 0.134
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 8


In [ ]:
# Ô 2 — Chia train/test và khóa tập test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,stratify=y, random_state=42)
print(len(X_train), len(X_test) , round(y_train.mean(),3), round(y_test.mean(),3))

4179 1393 0.134 0.134


In [ ]:
# Ô 3 — Cách làm ĐÚNG, viết tay từng bước
vec = TfidfVectorizer()
X_train_vec = vec.fit_transform(X_train) # HỌC vocabulary + IDF từ train, rồi biến đổi train
X_test_vec = vec.transform(X_test)         # CHỈ biến đổi test bằng vocabulary đã học
# test không được fit, chỉ được biến đổi bằng vocabulary của train. Từ nào trong test chưa xuất hiện ở train sẽ bị bỏ qua (đúng như Ô 5 Buổi 3).
print(X_train_vec.shape, X_test_vec.shape)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_vec,y_train)
print(round(clf.score(X_test_vec, y_test), 3)) 
print(round((y_test == 0).mean(), 3))  #  baseline: đoán toàn "không spam"

# Quy tắc vàng: fit chỉ trên train; trên test chỉ transform.

(4179, 7387) (1393, 7387)
0.972
0.866


In [8]:
# Ô 4 — Cách làm SAI: fit TF-IDF trên toàn bộ dữ liệu trước khi chia
vec_all = TfidfVectorizer().fit(X)          # SAI: học từ CẢ 5.572 tin, gồm cả tập test
X_train_w = vec_all.transform(X_train)
X_test_w = vec_all.transform(X_test)

clf_w = LogisticRegression(max_iter=1000).fit(X_train_w, y_train)
print(X_train_w.shape, round(clf_w.score(X_test_w, y_test), 3))
# (4179, 8713) 0.962

(4179, 8713) 0.962


In [ ]:
# Ô 5 — Cụ thể cái gì đã bị rò rỉ
only_test = set(vec_all.vocabulary_) - set(vec.vocabulary_)
# phép hiệu tập hợp, ra những từ có trong vocabulary “sai” mà không có trong vocabulary “đúng”, tức các từ chỉ xuất hiện trong tập test.
print(len(only_test), sorted(only_test)[:6])
# 1326 ['0125698789', '07090201529', '07090298926', '0776xxxxxxx', '07880867867', '07946746291']

idf_dung = dict(zip(vec.get_feature_names_out(), vec.idf_))
idf_sai = dict(zip(vec_all.get_feature_names_out(), vec_all.idf_))
print(round(idf_dung["free"], 3), round(idf_sai["free"], 3))   # 4.22 4.188

1326 ['0125698789', '07090201529', '07090298926', '0776xxxxxxx', '07880867867', '07946746291']
4.22 4.188


In [ ]:
# Ô 6 — Pipeline: gói các bước để không thể làm sai
pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000)),
])
# Pipeline([("tên", bước), ...]): danh sách các bước, mỗi bước có một tên do bạn đặt. Bước cuối là mô hình, các bước trước là tiền xử lý.
pipe.fit(X_train, y_train)
# pipe.fit(X_train, y_train) tự làm
# tfidf.fit_transform(X_train): học vocabulary chỉ từ train.
# clf.fit(...): học mô hình trên ma trận vừa tạo.
print(round(pipe.score(X_test, y_test), 3))   # 0.972  <- giống hệt cách đúng ở Ô 3
# pipe.score(X_test, y_test) tự làm: tfidf.transform(X_test) (không fit), rồi clf.score(...). Kết quả 0,972, giống hệt Ô 3, vì Pipeline làm đúng y như cách viết tay.

moi = ["WIN a FREE prize! Call 0800 now to claim", "Are we still meeting for lunch today?"]
print(pipe.predict(moi))                  # [1 0]
# pipe.predict(moi): đưa chữ thô vào, nhận nhãn ra. Không cần tự gọi vectorizer
print(pipe.predict_proba(moi).round(3))   # [[0.025 0.975] [0.961 0.039]]
# [[0.025 0.975]     ← tin 1: 2.5% ham, 97.5% spam
#  [0.97 0.03]]    ← tin 2: 97% ham, 3% spam
# pipe.predict_proba(...) trả về xác suất mô hình gán cho từng lớp, thay vì chỉ một đáp án cuối cùng.

print(len(pipe.named_steps["tfidf"].vocabulary_), pipe.named_steps["clf"].coef_.shape)
# pipe.named_steps["tfidf"]: truy cập từng bước bên trong bằng tên đã đặt, để xem vocabulary hay hệ số.
# 7387 (1, 7387)

tfidf = pipe.named_steps["tfidf"]
clf   = pipe.named_steps["clf"]

coef = pd.Series(clf.coef_[0], index=tfidf.get_feature_names_out()).sort_values()

print("Từ nghiêng về HAM :", coef.head(10).round(2).to_dict())
print("Từ nghiêng về SPAM:", coef.tail(10).round(2).to_dict())

0.972
[1 0]
[[0.025 0.975]
 [0.961 0.039]]
7387 (1, 7387)
[ 0.61494177  0.95456652 -0.01396124 ... -0.02199841  0.11860022
 -0.01096566]
Từ nghiêng về HAM : {'me': -2.4, 'my': -2.25, 'll': -1.6, 'that': -1.57, 'ok': -1.57, 'gt': -1.54, 'lt': -1.53, 'but': -1.44, 'can': -1.39, 'at': -1.39}
Từ nghiêng về SPAM: {'claim': 2.76, 'www': 2.86, 'uk': 2.89, 'reply': 2.9, 'to': 2.9, 'text': 2.94, 'free': 3.11, 'stop': 3.12, 'call': 3.93, 'txt': 4.31}


In [15]:
# Ô 7 — Pipeline thực sự cần thiết khi dùng cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ĐÚNG: đưa chữ thô + Pipeline vào cross-validation
s_dung = cross_val_score(pipe, X_train, y_train, cv=cv)
# Cách đúng: cross_val_score nhận pipe và chữ thô. Ở mỗi vòng, nó gọi pipe.fit trên 4 nhóm “học”, nên TF-IDF chỉ học vocabulary từ 4 nhóm đó, còn nhóm “kiểm tra” chỉ được transform.
print(s_dung.round(4), round(s_dung.mean(), 4), "±", round(s_dung.std(), 4))
# [0.9605 0.9701 0.9653 0.9545 0.9689] 0.9639 ± 0.0057

# SAI: biến đổi TF-IDF trước trên cả X_train, rồi mới cross-validation
X_train_tf = TfidfVectorizer().fit_transform(X_train)
s_sai = cross_val_score(LogisticRegression(max_iter=1000), X_train_tf, y_train, cv=cv)
print(s_sai.round(4), round(s_sai.mean(), 4), "±", round(s_sai.std(), 4))
# [0.9533 0.9545 0.9581 0.9498 0.9641] 0.956 ± 0.0049

[0.9605 0.9701 0.9653 0.9545 0.9689] 0.9639 ± 0.0057
[0.9533 0.9545 0.9581 0.9498 0.9641] 0.956 ± 0.0049


In [16]:
# Ô 8 — Khi rò rỉ trở nên nghiêm trọng: dữ liệu ngẫu nhiên đạt 80%
rng = np.random.default_rng(0)
X_noise = rng.normal(size=(200, 5000))     # 200 mẫu, 5.000 feature hoàn toàn ngẫu nhiên
y_noise = rng.integers(0, 2, size=200)     # nhãn tung đồng xu: 107 số 0, 93 số 1
cv_n = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# SAI: chọn 20 feature "tốt nhất" trên TOÀN BỘ dữ liệu, rồi mới cross-validation
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
sai = cross_val_score(LogisticRegression(), X_sel, y_noise, cv=cv_n)
print(sai.round(3), round(sai.mean(), 3))   # [0.775 0.8 0.825 0.8 0.775] 0.795

# ĐÚNG: bước chọn feature nằm TRONG Pipeline
dung_pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression())
dung = cross_val_score(dung_pipe, X_noise, y_noise, cv=cv_n)
print(dung.round(3), round(dung.mean(), 3))   # [0.575 0.7 0.45 0.55 0.475] 0.55

print(round(cross_val_score(DummyClassifier(strategy="most_frequent"),
                            X_noise, y_noise, cv=cv_n).mean(), 3))   # 0.535

[0.775 0.8   0.825 0.8   0.775] 0.795
[0.575 0.7   0.45  0.55  0.475] 0.55
0.535


In [17]:
# Ô 9 — Vì sao cách sai đạt 80% trên dữ liệu ngẫu nhiên
corr = np.array([abs(np.corrcoef(X_noise[:, j], y_noise)[0, 1]) for j in range(5000)])
print(round(np.median(corr), 3), round(np.sort(corr)[-20], 3), round(corr.max(), 3))
# 0.049 0.2 0.305

0.049 0.2 0.305


In [ ]:
# Ô 10 — Mở Pipeline ra xem mô hình học được gì
vocab = pipe.named_steps["tfidf"].get_feature_names_out()
coef = pipe.named_steps["clf"].coef_[0]
he_so = pd.Series(coef, index=vocab).sort_values()

print(he_so.tail(6).round(2).to_dict())   # đẩy về SPAM mạnh nhất
# {'text': 2.94, 'free': 3.11, 'stop': 3.12, 'call': 3.93, 'txt': 4.31, ...}
print(he_so.head(8).round(2).to_dict())   # đẩy về TIN THƯỜNG mạnh nhất
# {'me': -2.4, 'my': -2.25, 'll': -1.6, 'that': -1.57, 'ok': -1.57, 'gt': -1.54, 'lt': -1.53, ...}